# Café agroclimático: ejecución completa en Colab (CPU)
Equipo: Jhon Jairo Pulgarin, Yesid Hurtado, Frank López. SI3003, grupo Jueves.

El LLM corre en NVIDIA por API. La descarga, agregación y entrenamiento de la red bayesiana se realizan en este runtime de Colab, nunca en tu PC. No se entrena ni descarga un LLM. CPU es suficiente; no actives GPU.

1. En tu PC, ejecuta `python scripts/package_colab.py` (solo empaqueta archivos).
2. Abre este notebook en Colab. Sube `artifacts/cafe_colab_source.zip` cuando lo solicite.
3. En el panel **Secretos** de Colab, crea `NVIDIA_API_KEY` y habilita el acceso del notebook. También se admite entrada oculta.
4. Ejecuta las celdas en orden. Al final descarga los resultados y el modelo.


In [ ]:
from google.colab import files
from pathlib import Path
from zipfile import ZipFile
import os, subprocess, sys
uploaded = files.upload()
source_zip = next((name for name in uploaded if name.endswith('.zip')), None)
if source_zip is None:
    raise ValueError('Sube cafe_colab_source.zip')
work = Path('/content/cafe-agroclimatico')
work.mkdir(exist_ok=True)
with ZipFile(source_zip) as z:
    for name in z.namelist():
        if not (work / name).resolve().is_relative_to(work.resolve()):
            raise ValueError('Ruta inválida en ZIP')
    z.extractall(work)
os.chdir(work)
profile = 'requirements-cloud.txt' if sys.version_info >= (3, 12) else 'requirements.txt'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', profile], check=True)
print('Código listo en', work)


In [ ]:
from google.colab import userdata
from getpass import getpass
try:
    key = userdata.get('NVIDIA_API_KEY')
except Exception:
    key = getpass('NVIDIA API key (entrada oculta): ')
if not key:
    raise ValueError('Se requiere una clave NVIDIA para evaluar el LLM real')
os.environ['LLM_API_KEY'] = key
os.environ['LLM_BASE_URL'] = 'https://integrate.api.nvidia.com/v1'
os.environ['LLM_MODEL'] = 'nvidia/nemotron-3.5-lightning-30b-a3b'
os.environ['RAG_BACKEND'] = 'tfidf'
del key

def run(module, *args):
    import subprocess, sys
    from pathlib import Path
    log = Path('results/logs') / (module.replace('.', '_') + '.log')
    log.parent.mkdir(parents=True, exist_ok=True)
    command = [sys.executable, '-u', '-m', module, *args]
    with log.open('w', encoding='utf-8') as saved:
        with subprocess.Popen(command, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True,
                              encoding='utf-8', errors='replace', bufsize=1) as proc:
            for line in proc.stdout:
                print(line, end='', flush=True)
                saved.write(line)
                saved.flush()
            code = proc.wait()
    if code:
        raise RuntimeError(f'{module} terminó con código {code}. Error completo arriba y en {log}.')
    print(f'Completado: {module}', flush=True)
run('cafe.verify_services')


## Datos reales y corpus
Rango fijo para reproducibilidad: 2022-01-01 a 2026-10-31. La descarga puede tomar varios minutos o más según Socrata. Tiene caché mensual: si falla una consulta, vuelve a ejecutar la celda. No reduzcas los filtros de calidad para forzar un resultado. Revisa el reporte de cobertura antes de entrenar.

Plan B: carga CSV diarios con el esquema del README en `data/raw/` y omite solamente la descarga. No reemplaces datos con los sintéticos de pruebas.

Si falla un paso, el error completo aparece en la celda y se conserva en results/logs/. Corrige la causa y reejecuta solo ese paso: la caché mensual existente no se borra.


In [ ]:
run('cafe.fetch_corpus')
run('cafe.data_download', '--desde', '2022-01-01', '--hasta', '2026-10-31')
run('cafe.data_prep')


In [ ]:
import pandas as pd, json
from IPython.display import display
weekly = pd.read_csv('data/processed/semanal.csv', parse_dates=['semana'])
display(pd.read_csv('results/cobertura_datos.csv'))
quality = json.loads(Path('results/calidad_datos.json').read_text())
print(json.dumps(quality, ensure_ascii=False, indent=2))
if quality['cobertura_humedad'] == 0:
    raise ValueError('Sin humedad válida: revisar la fuente antes de aceptar los datos para 3.5')
display(weekly.describe(include='all').T)


## Entrenamiento y evaluación
La red bayesiana aprende conteos/CPT; no necesita GPU ni fine-tuning del LLM. La regresión logística es solo referencia de librería. Las métricas del indicador climático no demuestran incidencia de roya en campo. Se preservan por separado las 12 preguntas del agente con LLM y con respaldo.


In [ ]:
run('cafe.risk_model')
run('cafe.evaluate_csp')
run('cafe.evaluate_agent', '--mock')
run('cafe.evaluate_agent')
run('cafe.check_delivery')


In [ ]:
from IPython.display import Markdown, display
for name in ['metricas_modelo.md', 'eval_csp.md', 'eval_agente_llm.md', 'eval_agente_mock.md']:
    display(Markdown(Path('results', name).read_text()))


## Guardar evidencia y usar el dashboard
Descarga el ZIP antes de cerrar el runtime. Contiene `data/processed`, `results`, corpus y manifiestos de los datos, sin claves. En tu PC, extrae esas carpetas en el proyecto y ejecuta `streamlit run app.py`: solo presenta los artefactos ya calculados; el LLM sigue remoto.

También puedes subir el repositorio y los artefactos a Streamlit Community Cloud y configurar la clave como secreto/variable de entorno. Ver `docs/EJECUCION_NUBE.md` para el procedimiento. No hace falta exponer Colab mediante túneles.


In [ ]:
from zipfile import ZIP_DEFLATED
Path('results/entorno.txt').write_text(subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True))
out = Path('/content/cafe_resultados.zip')
with ZipFile(out, 'w', ZIP_DEFLATED) as z:
    for folder in ['data/processed', 'results', 'corpus']:
        for path in Path(folder).rglob('*'):
            if path.is_file() and path.suffix not in {'.pyc', '.zip'}:
                z.write(path, path.as_posix())
    for path in Path('data/raw').glob('*.json'):
        z.write(path, path.as_posix())
files.download(str(out))
